In [1]:
import torch
inputs = torch.tensor(
    [[
        [0.43, 0.15, 0.89], # Your     
        [0.55, 0.87, 0.66], # journey  
        [0.57, 0.85, 0.64], # starts   
        [0.22, 0.58, 0.33], # with     
        [0.77, 0.25, 0.10], # one      
        [0.05, 0.80, 0.55]  # step     
    ],
    [
        [0.43, 0.15, 0.89], # His     
        [0.55, 0.87, 0.66], # journey  
        [0.57, 0.85, 0.64], # end   
        [0.22, 0.58, 0.33], # at     
        [0.77, 0.25, 0.10], # last      
        [0.05, 0.80, 0.55]  # way     
    ]
])

In [2]:

import torch.nn as nn

class MaskedSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias = False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.d_out = d_out
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer(
            "mask",
            torch.triu( torch.ones(context_length,context_length) , diagonal=1 )
        )
        
    def forward(self,x):
        no_of_batch, no_of_token, embedding_dim = x.shape
        
        
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        # attention = Q @ (K)T
        attention_score = queries @ keys.transpose(1,2)
        
        # masking
        attention_score = attention_score.masked_fill(
            self.mask.bool()[:no_of_token, :no_of_token], -torch.inf
        )
        
        # normalize
        d_k = keys.shape[-1]
        attention_weight = torch.softmax(
            attention_score/d_k**0.5, dim=-1
        )
        
        # dropout
        attention_weight = self.dropout(attention_weight)
        
        context_vector = attention_weight @ values
        return context_vector

In [3]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout,num_heads,qkv_bias = False):
        super().__init__()
        self.heads = nn.ModuleList(
            [MaskedSelfAttention(d_in,d_out,context_length,dropout,qkv_bias) for _ in range(num_heads)]
        )
        
    def forward(self, x):
        return torch.concat([head(x) for head in self.heads], dim=-1)

In [5]:
d_in = inputs.shape[-1]
d_out = 2
context_length = inputs.shape[1]
print(d_in, d_out, context_length)

3 2 6


In [6]:
mha = MultiHeadAttentionWrapper(d_in,d_out,context_length, dropout=0.0, num_heads=2)

In [7]:
context_vector = mha(inputs)
context_vector

tensor([[[-0.1010,  0.2905, -0.3474, -0.2094],
         [ 0.0520,  0.1904, -0.1995, -0.1052],
         [ 0.0994,  0.1505, -0.1511, -0.0711],
         [ 0.1288,  0.1233, -0.1049, -0.0434],
         [ 0.0583,  0.0580, -0.1038, -0.0433],
         [ 0.1098,  0.0757, -0.0774, -0.0269]],

        [[-0.1010,  0.2905, -0.3474, -0.2094],
         [ 0.0520,  0.1904, -0.1995, -0.1052],
         [ 0.0994,  0.1505, -0.1511, -0.0711],
         [ 0.1288,  0.1233, -0.1049, -0.0434],
         [ 0.0583,  0.0580, -0.1038, -0.0433],
         [ 0.1098,  0.0757, -0.0774, -0.0269]]], grad_fn=<CatBackward0>)

In [8]:
context_vector.shape

torch.Size([2, 6, 4])